# TF-IDF – Project Notebook

Use this notebook for carrying out the analyses from the workshop notebook on your own subreddit data.

### Icons Used in This Notebook
💭 **Reflection**: Reflecting on ethical implications, biases, and social impact in data science.<br>

## Retrieving the Dataset

In [ ]:
import os
import pandas as pd

In [ ]:
# Replace this with your own preprocessed file!    
df = pd.read_csv('../../data/YOUR_FILE_PP.csv')

# Make sure the index is reset
df.reset_index(drop=True, inplace=True)

In [ ]:
# Remove all rows that are '[removed]' or '[deleted]'
df = df.loc[~df['pp_text'].isin(['[removed]', '[deleted]' ]),:]

# Select only rows that have >3 characters in selftext
df = df.loc[df['pp_text'].str.len() > 3]

## Using TF-IDF on your data

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TfidfVectorizer parameter guide:
#   max_df=0.85    — ignore terms appearing in more than 85% of documents (corpus-wide noise)
#   max_features=1000 — keep only the top 1000 terms by corpus frequency (limits matrix size)
#   min_df=5       — ignore terms appearing in fewer than 5 documents (rare noise/typos)
#   decode_error='ignore' — silently drop any bytes that can't be decoded (handles encoding artifacts)
#   smooth_idf=True — add 1 to numerator/denominator in IDF formula to avoid zero-division
#   use_idf=True   — multiply TF by IDF; setting False gives raw TF counts instead
tfidf_vectorizer = TfidfVectorizer(max_df=0.85,
                                   max_features=1000,
                                   min_df=5,
                                   decode_error='ignore',
                                   stop_words='english',
                                   smooth_idf=True,
                                   use_idf=True)

# Fit and transform the texts
tfidf = tfidf_vectorizer.fit_transform(df['pp_text'])

Let's have a look at some of the TF-IDF values:

In [ ]:
# Place TF-IDF values in a DataFrame
feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame.sparse.from_spmatrix(tfidf, columns=feature_names)

In [ ]:
tfidf_df.reset_index(drop=True, inplace=True)

In [ ]:
tfidf_df.head()

In [ ]:
# Highest TF-IDF values across documents
tfidf_df.sum().sort_values(ascending=False)

## Top TF-IDF Terms per Post

Change `tfidf[10]` to another number if you want to see tf-idf counts for a different post.

In [ ]:
import numpy as np

def get_top_tfidf_words(row, features, top_n=10):
    top_indices = np.argsort(row)[::-1][:top_n]
    return [(features[i], row[i]) for i in top_indices]

# Example: document 10
top_words = get_top_tfidf_words(tfidf[10].toarray()[0], tfidf_vectorizer.get_feature_names_out())
for word, score in top_words:
    print(f"{word}: {score:.4f}")

Let's look at the post itself to see what terms TF-IDF is considering "distinctive".

In [ ]:
df.selftext[10]

We can visualize these TF-IDF-weighted terms as well. This code saves the plot in a PNG file.

In [ ]:
import matplotlib.pyplot as plt

def plot_top_terms(tfidf_vector, feature_names, doc_id=0, top_n=10):
    row = tfidf_vector[doc_id].toarray()[0]
    top_indices = row.argsort()[-top_n:][::-1]
    terms = [feature_names[i] for i in top_indices]
    scores = [row[i] for i in top_indices]

    plt.figure(figsize=(8, 5))
    plt.barh(terms[::-1], scores[::-1])
    plt.title(f"Top {top_n} TF-IDF Terms for Document {doc_id}")
    plt.xlabel("TF-IDF Score")
    plt.tight_layout()
    plt.savefig(f"outputs_project/top_terms_doc_{doc_id}.png", dpi=300)
    plt.show()

# Change doc_id below to get data for a different post
plot_top_terms(tfidf, tfidf_vectorizer.get_feature_names_out(), doc_id=10)

## Top Terms Across the Corpus (Mean TF-IDF)

Now, let's move from document-level to corpus-level views:

In [ ]:
mean_tfidf = tfidf.mean(axis=0).A1
terms = tfidf_vectorizer.get_feature_names_out()
top_indices = mean_tfidf.argsort()[-10:][::-1]

top_terms = [terms[i] for i in top_indices]
top_scores = [mean_tfidf[i] for i in top_indices]

plt.figure(figsize=(8, 5))
plt.barh(top_terms[::-1], top_scores[::-1])
plt.title("Top TF-IDF Terms Across Corpus")
plt.xlabel("Mean TF-IDF Score")
plt.tight_layout()
plt.savefig("outputs_project/top_terms_corpus.png", dpi=300)
plt.show()

## Visualizing Document Space with PCA

Each document is now a high-dimensional TF-IDF vector. We can't visualize that many dimensions directly, but **Principal Component Analysis (PCA)** can project that space down to 2 dimensions while preserving as much variance as possible. Documents that use similar vocabulary will appear close together.

💡 **Tip**: If your data has a categorical column (like a flair or label), check the lesson notebook to see how to color the points by group.

⚠️ **Warning**: PCA is a linear projection. It may not reveal all meaningful structure; t-SNE or UMAP (used in later weeks) can capture non-linear patterns that PCA misses. Also, `.toarray()` can use a lot of memory on large datasets — if this cell is slow, lower `max_features` in the vectorizer above.

In [ ]:
from sklearn.decomposition import PCA

# Reduce to 2 components for visualization
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(tfidf.toarray())

plt.figure(figsize=(9, 6))
plt.scatter(coords[:, 0], coords[:, 1], alpha=0.3, s=5)
plt.title("TF-IDF Document Space (PCA projection)")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.tight_layout()
plt.savefig("outputs_project/pca_tfidf.png", dpi=300)
plt.show()

print(f"Total variance explained: {sum(pca.explained_variance_ratio_)*100:.1f}%")

## 💭 Reflection
- What kinds of words does TF-IDF seem to prioritize, and which does it ignore? Are these terms really the most "important" in a post?  
- How might this weighting reinforce certain biases (e.g. technical terms, rare slang, moral judgments)?  
- What assumptions are we making about what matters in language?

## Using TF-IDF to find Similar Posts

Choose a post or comment from your data that has an interesting topic or tone. 

In [ ]:
doc_idx = 25

Cange this `selftext` column to `body` if you are working with a comments DataFrame!

In [ ]:
df['selftext'].iloc[doc_idx]

Let's have a quick look at the TF-IDF scores for the words in this submission to see if these words are indeed typical for this particular submission. Do the distinctive words have to do with the topic of the post?

In [ ]:
tfidf_df.loc[doc_idx].sort_values(ascending=False)

Now let's find the closest posts to this one.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(tfidf)
similarities.shape

Put the text and scores in a dataframe, and sort by the score:

In [ ]:
similar_df = pd.DataFrame({
    # Change this to "body" if working with comments
    'text': df['selftext'].values,
    'score': similarities[doc_idx]}).sort_values('score', ascending=False)

The top document will be the document itself (it's going to have a similarity of 1 with itself). So we look at the next document - does it seem similar?

In [ ]:
similar_df['text'].iloc[0]

In [ ]:
similar_df['text'].iloc[1]

💭 **Reflection**: Reading similar posts like this can help you expand your ideas about the **research question** you have about your data. For instance, you might find that the ideological concepts you are interested in are used in other contexts you hadn't previously considered. Or you might find "adjacent" concepts that give you a more robust understanding of the discourse of your community. 

## Using TF-IDF to Find Posts

💭 **Reflection**: Enter a term that makes sense given your dataset. It should be a term that says something about a dominant theme or topic you are expecting to find in your data. **Check out the output from `tfidf_df` above to see some some distinctive terms**.

The resulting DataFrame will be posts where the word has the greatest significance and specificity compared to the other posts.

If the resulting DataFrame is empty, lower the threshold from `.5` to something lower like `.3`.

In [ ]:
# Subsetting one DF with the mask of another DF
tfidf_someword_df = df[tfidf_df['SOME_TERM'] > .5]
tfidf_someword_df.head(3)

The first post from that DataFrame is the post in which your chosen word has the most significance – according to TF-IDF, at least.

In [ ]:
print(tfidf_someword_df['selftext'].iloc[0])

## Comparing Vocabulary Between Two Groups

One of the most powerful uses of TF-IDF is comparing the characteristic vocabulary of two groups of posts. If your data has a categorical column (like a flair, tag, or label), you can ask: **which words are most distinctive of each group?**

Run `list(df)` or `df['SOME_COLUMN'].value_counts()` to see which categorical columns your data has and what values they contain. Then replace `YOUR_CATEGORY_COLUMN`, `GROUP1_VALUE`, and `GROUP2_VALUE` in the code below.

NOTE: If your dataset has no categorical column to compare, you can skip this section.

🔔 **Question**: Before running the code, make a prediction. What kinds of words do you expect to be distinctive for each group?

In [ ]:
import numpy as np

# Replace these with your own column and two of its values!
category_col = 'YOUR_CATEGORY_COLUMN'
group1_value = 'GROUP1_VALUE'
group2_value = 'GROUP2_VALUE'

group1_mask = df[category_col] == group1_value
group2_mask = df[category_col] == group2_value

# Mean TF-IDF per group
group1_mean = tfidf[group1_mask.values].mean(axis=0).A1
group2_mean = tfidf[group2_mask.values].mean(axis=0).A1

features = tfidf_vectorizer.get_feature_names_out()

# Top 15 distinctive words per group (high mean TF-IDF)
top_n = 15
g1_top_idx = group1_mean.argsort()[-top_n:][::-1]
g2_top_idx = group2_mean.argsort()[-top_n:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh([features[i] for i in g1_top_idx[::-1]],
             [group1_mean[i] for i in g1_top_idx[::-1]], color='steelblue')
axes[0].set_title(f"Top Terms: {group1_value}")
axes[0].set_xlabel("Mean TF-IDF")

axes[1].barh([features[i] for i in g2_top_idx[::-1]],
             [group2_mean[i] for i in g2_top_idx[::-1]], color='tomato')
axes[1].set_title(f"Top Terms: {group2_value}")
axes[1].set_xlabel("Mean TF-IDF")

plt.tight_layout()
plt.savefig("outputs_project/group_comparison_tfidf.png", dpi=300)
plt.show()

print(f"{group1_value} posts: {group1_mask.sum()}  |  {group2_value} posts: {group2_mask.sum()}")

## 💭 Reflection: What Is TF-IDF Blind To?

TF-IDF treats text as a **bag of words** — a collection of tokens with no order, no context, and no syntax. It can tell you which words are statistically distinctive, but it cannot tell you:

- **Negation**: "I did *not* take the money" and "I did take the money" look nearly identical to TF-IDF.
- **Sarcasm and irony**: Common in online communities. A word like "obviously" has a very different meaning depending on tone.
- **Context of co-occurrence**: Whether "angry" appears near "husband" or near "myself" changes meaning completely — but TF-IDF can't see that.
- **Who is speaking**: The poster's identity, power position, or relationship to the people they describe shapes the story. TF-IDF counts words regardless of whose perspective they come from.

Looking at the group comparison you just made:

🔔 **Question**: Do the different vocabulary profiles reflect a real difference in how members of your community *narrate* their experiences? Or could they reflect a different kind of situation being described?

🔔 **Question**: Imagine you trained a classifier to predict the group label from TF-IDF features. What would it be learning, exactly? What would it not be learning?

## Using TF-IDF Correlations to Explore Biases

In [ ]:
corr = tfidf_df.corr()

💭 **Reflection**: Pick two words you are interested in comparing. Ideally, they should be binary constructs like "man" and "woman", or "progressive" and "conservative", or "Islam" and "Christianity". 

Change the `WORD1` and `WORD2` values to two binary concepts you are interested in comparing. Also change the `by=` argument to one of the words you picked.

The resulting DataFrame will be a list of words ordered by the amount of correlation with the word you picked when setting the `by=` argument. These correlations are a rough representation of other words that are frequently co-occurring with the word you picked.

In [ ]:
# Replace WORD1 and WORD2 with your two words!
corr[['WORD1','WORD2']].sort_values(by='WORD2',ascending=False)[:30]

## 💭 Reflection: 

- Do these related terms make sense? 
- Do you see some terms that could be indicative of a bias towards a binary construct in the data? 